In [1]:
import pandas as pd

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(
    "Motor_Vehicle_Collisions_-_Crashes_20260411.csv",
    dtype=str,          # read everything as string first to avoid parse issues
    low_memory=False
)

# Normalize column names (strip whitespace, lowercase)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# ── Filter 1: Manhattan only ──────────────────────────────────────────────────
df_manhattan = df[df["borough"].str.strip().str.upper() == "MANHATTAN"].copy()
print(f"Total rows:         {len(df):,}")
print(f"Manhattan rows:     {len(df_manhattan):,}")

# ── Filter 2: Alcohol involvement in ANY of the 5 contributing factor cols ────
contrib_cols = [
    "contributing_factor_vehicle_1",
    "contributing_factor_vehicle_2",
    "contributing_factor_vehicle_3",
    "contributing_factor_vehicle_4",
    "contributing_factor_vehicle_5",
]

alcohol_mask = df_manhattan[contrib_cols].apply(
    lambda col: col.str.strip().str.lower() == "alcohol involvement"
).any(axis=1)

df_alcohol = df_manhattan[alcohol_mask].copy()
print(f"Alcohol-involved:   {len(df_alcohol):,}")

# ── Missing lat/lon report ────────────────────────────────────────────────────
df_alcohol["latitude"]  = pd.to_numeric(df_alcohol["latitude"],  errors="coerce")
df_alcohol["longitude"] = pd.to_numeric(df_alcohol["longitude"], errors="coerce")

missing_lat  = df_alcohol["latitude"].isna().sum()
missing_lon  = df_alcohol["longitude"].isna().sum()
missing_both = df_alcohol[df_alcohol["latitude"].isna() & df_alcohol["longitude"].isna()]
total        = len(df_alcohol)

print(f"\n── Missing lat/lon report ──────────────────────────────")
print(f"Missing latitude:       {missing_lat:,}  ({missing_lat/total*100:.1f}%)")
print(f"Missing longitude:      {missing_lon:,}  ({missing_lon/total*100:.1f}%)")
print(f"Missing both:           {len(missing_both):,}  ({len(missing_both)/total*100:.1f}%)")

# ── What do the missing rows look like? ───────────────────────────────────────
print("\n── Sample of rows missing lat/lon ──────────────────────")
print(missing_both[[
    "crash_date", "crash_time", "on_street_name",
    "cross_street_name", "off_street_name", "zip_code", "borough"
]].head(15).to_string(index=False))

# ── Do missing rows at least have street names we could geocode? ──────────────
has_on_street  = missing_both["on_street_name"].notna().sum()
has_cross      = missing_both["cross_street_name"].notna().sum()
has_off        = missing_both["off_street_name"].notna().sum()

print(f"\nOf the {len(missing_both):,} missing-coord rows:")
print(f"  Have on_street_name:    {has_on_street:,}")
print(f"  Have cross_street_name: {has_cross:,}")
print(f"  Have off_street_name:   {has_off:,}")
print(f"  Have none of the above: {(missing_both[['on_street_name','cross_street_name','off_street_name']].isna().all(axis=1)).sum():,}")

Total rows:         2,254,014
Manhattan rows:     346,125
Alcohol-involved:   2,951

── Missing lat/lon report ──────────────────────────────
Missing latitude:       73  (2.5%)
Missing longitude:      73  (2.5%)
Missing both:           73  (2.5%)

── Sample of rows missing lat/lon ──────────────────────
crash_date crash_time               on_street_name cross_street_name                 off_street_name zip_code   borough
03/26/2022       5:43                 GRAND STREET      essex street                             NaN    10002 MANHATTAN
09/22/2021      16:02              WEST 145 STREET          BROADWAY                             NaN    10031 MANHATTAN
07/24/2021       2:36 FREDERICK DOUGLASS BOULEVARD   WEST 142 STREET                             NaN    10030 MANHATTAN
08/01/2021       1:52                          NaN               NaN       805       RIVERSIDE DRIVE    10032 MANHATTAN
10/18/2021      21:39         SAINT NICHOLAS PLACE   WEST 153 STREET                           

In [2]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import socket

geolocator = Nominatim(user_agent="dont_crash_or_burn_project", timeout=10)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.5,
    max_retries=3,
    error_wait_seconds=5.0
)

def build_address(row):
    """
    Priority 1: intersection (on_street + cross_street) — most reliable
    Priority 2: off_street_name (already a full address like '805 RIVERSIDE DRIVE')
    """
    on  = str(row["on_street_name"]).strip()  if pd.notna(row["on_street_name"])  else ""
    cross = str(row["cross_street_name"]).strip() if pd.notna(row["cross_street_name"]) else ""
    off = str(row["off_street_name"]).strip()  if pd.notna(row["off_street_name"])  else ""

    if on and cross:
        return f"{on} & {cross}, Manhattan, New York, NY"
    elif off:
        return f"{off}, Manhattan, New York, NY"
    return None

def geocode_row(row):
    addr = build_address(row)
    if not addr:
        return None, None
    try:
        loc = geocode(addr)
        if loc:
            return loc.latitude, loc.longitude
    except Exception:
        pass
    return None, None

# ── Only run on the missing rows ──────────────────────────────────────────
missing_idx = df_alcohol[df_alcohol["latitude"].isna()].index

print(f"Geocoding {len(missing_idx)} rows — ~{len(missing_idx)} seconds at 1 req/sec...")

for idx in missing_idx:
    lat, lon = geocode_row(df_alcohol.loc[idx])
    df_alcohol.at[idx, "latitude"]  = lat
    df_alcohol.at[idx, "longitude"] = lon

# ── Report results ────────────────────────────────────────────────────────────
still_missing = df_alcohol["latitude"].isna().sum()
recovered     = len(missing_idx) - still_missing

print(f"\nRecovered:     {recovered} / {len(missing_idx)}")
print(f"Still missing: {still_missing}")

# ── Save clean file ───────────────────────────────────────────────────────────
df_clean = df_alcohol.dropna(subset=["latitude", "longitude"]).copy()
df_clean.to_csv("crash_manhattan_alcohol_clean.csv", index=False)
print(f"\nSaved {len(df_clean):,} rows → crash_manhattan_alcohol_clean.csv")

Geocoding 73 rows — ~73 seconds at 1 req/sec...


KeyboardInterrupt: 

In [ ]:
# Test one address manually to see what Nominatim actually returns
test_row = df_alcohol[df_alcohol["latitude"].isna()].iloc[0]
addr = build_address(test_row)
print("Address sent to Nominatim:", addr)

loc = geolocator.geocode(addr)
print("Result:", loc)

# If that returns None, try a simpler version
loc2 = geolocator.geocode("Grand Street & Essex Street, Manhattan, NY")
print("Simplified result:", loc2)

In [ ]:
#sanity check
df_clean = pd.read_csv("crash_manhattan_alcohol_clean.csv", dtype=str, low_memory=False)
df_clean["latitude"]  = pd.to_numeric(df_clean["latitude"],  errors="coerce")
df_clean["longitude"] = pd.to_numeric(df_clean["longitude"], errors="coerce")

print(f"Final clean rows:   {len(df_clean):,}")
print(f"Still missing:      {df_clean['latitude'].isna().sum()}")

# Sanity check: are all coords actually in Manhattan?
in_bounds = (
    df_clean["latitude"].between(40.70, 40.88) &
    df_clean["longitude"].between(-74.02, -73.91)
)
print(f"Coords in Manhattan bounds: {in_bounds.sum():,}")
print(f"Coords OUT of bounds:       {(~in_bounds).sum():,}")

# Peek at any out-of-bounds rows
print(df_clean[~in_bounds][["crash_date","latitude","longitude","on_street_name"]].head(10))

In [ ]:
# Drop (0.0, 0.0) placeholders and Rochester misfires
df_clean = df_clean[
    df_clean["latitude"].between(40.70, 40.88) &
    df_clean["longitude"].between(-74.02, -73.91)
].copy()

print(f"Final usable rows: {len(df_clean):,}")

# Save over the previous clean file
df_clean.to_csv("crash_manhattan_alcohol_clean.csv", index=False)
print("Saved → crash_manhattan_alcohol_clean.csv")

In [ ]:
df_clean.tail()

In [3]:
import h3
import pandas as pd

# ── Load clean file ───────────────────────────────────────────────────────────
df = pd.read_csv("crash_manhattan_alcohol_clean.csv", low_memory=False)
df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# ── Assign H3 zone ────────────────────────────────────────────────────────────
# Resolution 10 ≈ 70m per hex ≈ 2-block radius in Manhattan
H3_RES = 10

df["zone_id"] = df.apply(
    lambda r: h3.latlng_to_cell(r["latitude"], r["longitude"], H3_RES),
    axis=1
)

# ── Sanity check ──────────────────────────────────────────────────────────────
print(f"Total rows:         {len(df):,}")
print(f"Unique zones:       {df['zone_id'].nunique():,}")
print(f"Missing zone_id:    {df['zone_id'].isna().sum()}")

# Top 10 hotspot zones by crash count
print("\n── Top 10 zones by crash count ─────────────────────────")
print(df.groupby("zone_id").size().sort_values(ascending=False).head(10))

# ── Keep only the columns your MapReduce jobs actually need ───────────────────
mr_cols = [
    "zone_id",
    "crash_date",
    "crash_time",
    "number_of_persons_injured",
    "number_of_persons_killed",
    "number_of_pedestrians_injured",
    "number_of_pedestrians_killed",
    "number_of_cyclist_injured",
    "number_of_cyclist_killed",
    "number_of_motorist_injured",
    "number_of_motorist_killed",
    "contributing_factor_vehicle_1",
    "contributing_factor_vehicle_2",
    "contributing_factor_vehicle_3",
    "contributing_factor_vehicle_4",
    "contributing_factor_vehicle_5",
    "latitude",
    "longitude",
]

df_mr = df[mr_cols].copy()
df_mr.to_csv("crash_zones_mr_ready.csv", index=False)
print(f"\nSaved {len(df_mr):,} rows → crash_zones_mr_ready.csv")

Total rows:         2,887
Unique zones:       1,488
Missing zone_id:    0

── Top 10 zones by crash count ─────────────────────────
zone_id
8a2a100a128ffff    20
8a2a100d369ffff    14
8a2a1072ca0ffff    12
8a2a1072ca67fff    11
8a2a100a12f7fff    11
8a2a1072cba7fff    10
8a2a1072ca77fff    10
8a2a100ae75ffff     9
8a2a1072caaffff     9
8a2a1008d52ffff     9
dtype: int64

Saved 2,887 rows → crash_zones_mr_ready.csv


In [4]:
zone_counts = df.groupby("zone_id").size()

print("── Zone crash count distribution ───────────────────────")
print(f"Zones with only 1 crash:  {(zone_counts == 1).sum():,}  ({(zone_counts == 1).sum()/len(zone_counts)*100:.1f}%)")
print(f"Zones with 2-4 crashes:   {zone_counts.between(2,4).sum():,}")
print(f"Zones with 5-9 crashes:   {zone_counts.between(5,9).sum():,}")
print(f"Zones with 10+ crashes:   {(zone_counts >= 10).sum():,}")

── Zone crash count distribution ───────────────────────
Zones with only 1 crash:  807  (54.2%)
Zones with 2-4 crashes:   592
Zones with 5-9 crashes:   82
Zones with 10+ crashes:   7


In [5]:
import h3
import folium
import pandas as pd
from folium.plugins import HeatMap

df = pd.read_csv("crash_zones_mr_ready.csv")

# ── Aggregate crash counts per zone ──────────────────────────────────────────
zone_counts = df.groupby("zone_id").agg(
    crash_count   = ("zone_id", "count"),
    avg_lat       = ("latitude", "mean"),
    avg_lon       = ("longitude", "mean"),
    total_injured = ("number_of_persons_injured", "sum"),
    total_killed  = ("number_of_persons_killed",  "sum"),
).reset_index()

# ── Base map centered on Manhattan ────────────────────────────────────────────
m = folium.Map(
    location=[40.7831, -73.9712],
    zoom_start=13,
    tiles="CartoDB dark_matter"   # dark tile looks great for crash heatmaps
)

# ── Layer 1: H3 hex polygons colored by crash count ──────────────────────────
max_crashes = zone_counts["crash_count"].max()

def crash_color(count):
    """Red intensity scaled to crash count."""
    ratio = count / max_crashes
    if ratio > 0.75: return "#ff0000"
    if ratio > 0.50: return "#ff6600"
    if ratio > 0.25: return "#ff9900"
    return "#ffcc00"

for _, row in zone_counts.iterrows():
    # h3.cell_to_boundary returns (lat, lon) tuples — folium wants [lat, lon]
    boundary = h3.cell_to_boundary(row["zone_id"])
    boundary_latlon = [[lat, lon] for lat, lon in boundary]

    folium.Polygon(
        locations=boundary_latlon,
        color=crash_color(row["crash_count"]),
        fill=True,
        fill_color=crash_color(row["crash_count"]),
        fill_opacity=0.5,
        weight=0.5,
        tooltip=folium.Tooltip(
            f"""
            <b>Zone:</b> {row['zone_id']}<br>
            <b>Crashes:</b> {row['crash_count']}<br>
            <b>Injured:</b> {int(row['total_injured'])}<br>
            <b>Killed:</b>  {int(row['total_killed'])}
            """
        )
    ).add_to(m)

# ── Layer 2: Heatmap overlay for extra density visibility ────────────────────
heat_data = [
    [row["avg_lat"], row["avg_lon"], row["crash_count"]]
    for _, row in zone_counts.iterrows()
]
HeatMap(heat_data, radius=18, blur=15, min_opacity=0.3).add_to(m)

# ── Layer 3: Individual crash points (toggle-able) ───────────────────────────
crash_layer = folium.FeatureGroup(name="Individual crashes", show=False)
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        color="#ffffff",
        fill=True,
        fill_opacity=0.6,
        tooltip=f"{row['crash_date']} {row['crash_time']}"
    ).add_to(crash_layer)
crash_layer.add_to(m)

# ── Layer control (toggle layers on/off) ─────────────────────────────────────
folium.LayerControl().add_to(m)

# ── Save ──────────────────────────────────────────────────────────────────────
m.save("crash_heatmap.html")
print("Saved → crash_heatmap.html — open in any browser")

Saved → crash_heatmap.html — open in any browser
